<a href="https://colab.research.google.com/github/fre-denni/genAIforUX-research/blob/main/logbook_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install libraries

In [ ]:
!pip install -q pymupdf pdf2image transformers==4.44.2 google-genai spacy pandas scikit-learn pillow openpyxl
!python -m spacy download en_core_web_sm
!pip install timm einops
!pip install flash_attn

We are using an older version of transformers from hugging face to be sure that it's compatible with the Florence2 model from microsoft

### Import necessary libraries

In [ ]:
import os
import fitz  # PyMuPDF
import pandas as pd
from google import genai
from google.genai import types
import spacy
from transformers import AutoProcessor, AutoModelForCausalLM
import transformers.dynamic_module_utils as dynamic_utils
from PIL import Image
import torch
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from IPython.display import clear_output
clear_output()
print("Environment setup correctly!")

Setup folder



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define the base folder
base_folder = "/content/drive/MyDrive/logbook analysis/"

# Define input and output folders
input_folder = os.path.join(base_folder, "logbook")
output_folder = os.path.join(base_folder, "output")
output_images_folder = os.path.join(base_folder, "output_images")

# Create output folders if they don't exist
os.makedirs(output_folder, exist_ok=True)
os.makedirs(output_images_folder, exist_ok=True)

print(f"Base folder: {base_folder}")
print(f"Input folder set to: {input_folder}")
print(f"Output folder created/verified: {output_folder}")
print(f"Output images folder created/verified: {output_images_folder}")


Add your API

In [ ]:
from google.colab import userdata
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("API configurata")
except Exception as e:
  print("Errore nel caricamento")
  print(f"errore: {e}")

Upload and define function of Florence-2

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Caricamento Florence-2 su: {device}...")

florence_model_id = "microsoft/Florence-2-large"
processor = AutoProcessor.from_pretrained(florence_model_id, trust_remote_code=True)

# TRUCCO: Bendiamo il parser di Hugging Face per fargli ignorare 'flash_attn'
original_get_imports = dynamic_utils.get_imports
def custom_get_imports(filename):
    imports = original_get_imports(filename)
    return [imp for imp in imports if imp != "flash_attn"]

dynamic_utils.get_imports = custom_get_imports # Applichiamo la benda

# Carichiamo il modello in sicurezza (userà l'algoritmo standard)
florence_model = AutoModelForCausalLM.from_pretrained(florence_model_id, trust_remote_code=True).to(device).eval()

dynamic_utils.get_imports = original_get_imports # Togliamo la benda (Best practice)

def execute_florence(task_prompt, image):
    inputs = processor(text=task_prompt, images=image, return_tensors="pt").to(device)
    generated_ids = florence_model.generate(
      input_ids=inputs["input_ids"],
      pixel_values=inputs["pixel_values"],
      max_new_tokens=1024,
      early_stopping=False,
      do_sample=False,
      num_beams=3,
    )
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = processor.post_process_generation(
        generated_text,
        task=task_prompt,
        image_size=(image.width, image.height)
    )
    return parsed_answer

from IPython.display import clear_output
clear_output()

print("Florence-2 ready!")

Configure spacy

In [ ]:
nlp = spacy.load("en_core_web_sm") # English language
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Add custom rules for design methodologies
patterns = [
    # A1 - User Journey Maps & varianti
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": "journey"}, {"LOWER": {"IN": ["map", "maps"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "ujm"}]},

    # A1 -Storyboards
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["storyboard", "storyboards"]}}]},

    # A2 - Mental Model Diagrams (cattura sia "mental model" che "mental model diagram")
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mental"}, {"LOWER": "model"}, {"LOWER": {"IN": ["diagram", "diagrams"]}, "OP": "?"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mmd"}]},

    # A2 -Mind & Empathy Maps
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mind"}, {"LOWER": {"IN": ["map", "maps"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "empathy"}, {"LOWER": {"IN": ["map", "maps"]}}]},

    # A1 - User Jobs & JTBD
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": {"IN": ["job", "jobs"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "jobs"}, {"LOWER": "to"}, {"LOWER": "be"}, {"LOWER": "done"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "jtbd"}]},

    # A1 - User Stories
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": {"IN": ["story", "stories"]}}]},

    # A1 - Task Analysis (cattura sia "task analysis" che "task analysis diagram")
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "task"}, {"LOWER": "analysis"}, {"LOWER": {"IN": ["diagram", "diagrams"]}, "OP": "?"}]},

    # A2 - Concept / Conceptual Maps
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["concept", "conceptual"]}}, {"LOWER": {"IN": ["map", "maps"]}}]},

    # A3 - Research
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "literature"}, {"LOWER": "review"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "benchmarking"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["persona", "personas"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "digital"}, {"LOWER": "ethnography"}]},

    # A4 - Mapping (System, Service, Stakeholder) - cattura map, maps e mapping
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "system"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "service"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "stakeholder"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},

    # A2 - Affinity Diagram
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "affinity"}, {"LOWER": {"IN": ["diagram", "diagrams"]}}]}
]
ruler.add_patterns(patterns)

ruler.add_patterns(patterns)
print("Added custom config to spacy")

### Global function and prompts

Set up and force the JSON analysis for Gemini, and the various prompts for the tasks.

In [ ]:
# Force the config of output as json
json_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0 # more deterministic outputs
)



In [ ]:
# define the element present in the logbooks
def classify_assignment(page_text):
  prompt = f"""You are an expert analyst of student logbooks from a UX Design course.
    Your task is to analyze the OCR text extracted from a single page and determine which Assignment bucket it belongs to.

    Use the following ALLOWED CATEGORIES. I have listed the specific UX methodologies that belong to each category.
    Choose ONLY ONE category based on the presence of these keywords or explicit assignment titles:

    - 'A1': Temporal/Chronological maps.
      Keywords: Assignment 1, A1, Assignment 01,  Chrono Maps, User Journey Map, UJM, Storyboard, User Jobs, Jobs to be done, JTBD, User Stories, Task Analysis.

    - 'A2': Non-temporal/Conceptual maps.
      Keywords: Assignment 2, A2, Assignment 02, Mental Model Diagram, MMD, Mind Map, Empathy Map, Concept Map, Conceptual Map, Affinity Diagram.

    - 'A3': Research and Divergent mapping.
      Keywords: Assignment 3, A3, Assignment 03, Literature Review, Benchmarking, Personas, Digital Ethnography, Client Research.

    - 'A4': System and Service Mapping.
      Keywords: Assignment 4, A4, Assignment 04, System Map, System Mapping, Service Map, Service Blueprint, Stakeholder Map, App Mockup.

    - 'AI': Use of AI and Critical Reflections.
      Keywords: The use of AI, Reflections, Prompting, AI usage.
      *CRITICAL RULE: This category has TOP PRIORITY. If the main focus of the text is discussing the interaction with AI tools, prompting strategies, or reflecting on AI outputs, you MUST classify it as 'AI', even if it mentions A1, A2, A3, or A4.*

    - 'Intro': Structural pages.
      Keywords: Index, Table of Contents, Cover Page, Acknowledgments, Logbook Introduction.

    - 'Unknown': Use only if the text is illegible, missing, or completely out of context.

    PAGE TEXT TO ANALYZE:
    ---
    {page_text}
    ---

    Return ONLY a valid JSON object with this exact structure:
    {{
        "assignment": "CategoryName",
        "reasoning": "Brief reason in 10 words or fewer",
        "confidence": "High, Medium, or Low"
    }}
    """

  try:
      response = client.models.generate_content(
          model='gemini-1.5-flash',
          contents=prompt,
          config=json_config
      )
      return json.loads(response.text)
  except Exception as e:
      print(f"Gemini API Error: {e}")
      return {
          "assignment": "Unknown",
          "reasoning": "API Error",
          "confidence": "Low"
      }

---

# And now the OCR loop!



In [ ]:
def process_pdf_pipeline(pdf_path, output_dir="output_images"):
  #Extract the students ID
  student_id = os.path.basename(pdf_path).replace(".pdf", "")
  doc = fitz.open(pdf_path)

  master_records=[]

  print(f"Processing {student_id} ({len(doc)} pages)...")

  for page_num in range(len(doc)):
    print(f"Processing page {page_num + 1}...")

    #convert page in image for florence2
    page=doc.load_page(page_num)
    pix=page.get_pixmap(dpi=200) #good balance of quality and pixel weight
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    #object detection with florence2
    #identify  text, titles, images
    layout_results = execute_florence('<OD>', img)

    #extract coordinate and labels in a safe manner
    od_data = layout_results.get('<OD>', {})
    bboxes = od_data.get('bboxes', [])
    labels_list = od_data.get('labels', []) # Renamed to avoid confusion

    page_text_complete = ""
    records_current_page = []

    #Cut or OCR
    for idx, (bbox, label) in enumerate(zip(bboxes, labels_list)): # Iterate over label (singular)
      #Cut too little boxes or invalides
      if bbox[2] - bbox[0] < 10 or bbox[3] - bbox[1] < 10: continue
      crop_img = img.crop(bbox)

      #create filename
      clean_label = label.replace(" ", "_").lower() # Fixed: Use 'label' instead of 'labels'
      filename = f"{student_id}_pag{page_num+1}_{clean_label}_{idx}.png"
      filepath = os.path.join(output_dir, filename)
      crop_img.save(filepath)

      ocr_text=""
      #If text or title, let's execture OCR with florence2
      if clean_label in ['text', 'title']:
        ocr_results = execute_florence('<OCR>', crop_img)
        ocr_text = ocr_results.get('<OCR>', "")
        page_text_complete += ocr_text + "\n"

      #Save the temporary record
      records_current_page.append({
          "ID": student_id,
          "Page": page_num + 1, # Fixed: changed 'page_nume' to 'page_num'
          "Layout": clean_label,
          "OCR_Text": ocr_text,
          "Path_Image": filepath,
          "Assignment": "TBD"
      })

    # Classify Assignment
    assignment_label = "Unknown"
    if page_text_complete.strip():
      gemini_classification = classify_assignment(page_text_complete)
      assignment_label = gemini_classification.get("assignment", "Unknown")

    # Apply assignment to all elements in the page
    for record in records_current_page:
      record["Assignment"] = assignment_label
      master_records.append(record)

  print(f"Complete PDF {student_id}")

  #create master dataframe
  df_master = pd.DataFrame(master_records)
  return df_master


Quick prova

In [ ]:
import os

proof = os.path.join(input_folder, "11167713.pdf") #quick and dirty 3mb logbook

if os.path.exists(proof):
  indexed = process_pdf_pipeline(proof, output_dir=output_images_folder)

  # Ensure the 'results' directory exists before saving the CSV
  os.makedirs("results", exist_ok=True)

  #Save the result for security
  indexed.to_csv("results/master_index.csv", index=False)

  #Show preview
  display(indexed.head(100))
else:
  print(f"File not founded: {proof}")